# Basic analysis of mtg card data

Using the json provided by scryfall. starting @ 3/4/2026

In [177]:
import seaborn as sns
import pandas as pd
# import numpy as np


First we load the json of all cards with unique oracle identification numbers. So we are only getting one object per unique card, not unique printings.

In [178]:
cards = pd.read_json('oracle-cards-20260305100222.json')

Now we can get rid of the unnecessary columns. Think, the artist and whether or not it is a showcase or a foil.

In [179]:
# detail the columns that we are going to drop from the list because they
to_drop = ['id', 'multiverse_ids', 'mtgo_id',
       'tcgplayer_id', 'cardmarket_id', 'lang', 'released_at', 'uri',
       'scryfall_uri', 'layout', 'highres_image', 'image_status', 'image_uris',
       'games', 'reserved', 'foil', 'nonfoil', 'finishes',
       'oversized', 'promo', 'reprint', 'variation', 'set_uri', 'set_search_uri', 'scryfall_set_uri',
       'rulings_uri', 'prints_search_uri', 'collector_number', 'flavor_text', 'artist',
       'artist_ids', 'illustration_id', 'border_color', 'frame',
       'frame_effects', 'security_stamp', 'full_art', 'textless', 'booster',
       'story_spotlight', 'preview', 'related_uris',
       'purchase_uris', 'mtgo_foil_id', 'penny_rank', 'set_name', 'arena_id',
       'promo_types', 'tcgplayer_etched_id', 'attraction_lights', 'color_indicator', 'flavor_name',
       'content_warning', 'printed_type_line', 'defense', 'set_id','resource_id', 'digital', 'life_modifier', 'hand_modifier', ]

# drop the columns detailed above
simplified_cards = cards[cards['digital'] == False]
simplified_cards = cards.drop(to_drop, axis=1)


# set the index to be the oracle_id for consistency
simplified_cards.set_index("oracle_id", inplace=True)

Now we drop each column where the card does not have an edhrecrank and convert some of the columns to simpler 

In [180]:
simplified_cards = simplified_cards[simplified_cards["edhrec_rank"] > 0]

simplified_cards.info

simplified_cards.info()

print(simplified_cards["object"].unique())
simplified_cards["all_parts"].explode().explode().unique()

<class 'pandas.core.frame.DataFrame'>
Index: 30722 entries, 00037840-6089-42ec-8c5c-281f9f474504 to ffff90c3-63c4-4dee-a21d-6b2b113f4f80
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   object          30722 non-null  object 
 1   name            30722 non-null  object 
 2   mana_cost       30240 non-null  object 
 3   cmc             30722 non-null  float64
 4   type_line       30722 non-null  object 
 5   oracle_text     29963 non-null  object 
 6   power           16905 non-null  object 
 7   toughness       16905 non-null  object 
 8   colors          30240 non-null  object 
 9   color_identity  30722 non-null  object 
 10  keywords        30722 non-null  object 
 11  all_parts       5117 non-null   object 
 12  legalities      30722 non-null  object 
 13  game_changer    30722 non-null  bool   
 14  set             30722 non-null  object 
 15  set_type        30722 non-null  object 
 16  rarity         

array(['object', 'id', 'component', 'name', 'type_line', 'uri', nan],
      dtype=object)

Cast the columns with known possible conversions

In [181]:
# String columns first
string_conversion_targets = ['power', 'toughness', 'name', 'type_line', 'oracle_text', 
                              'set', 'set_type', 'watermark', 'rarity', 'printed_text']
for target in string_conversion_targets:
    simplified_cards[target] = simplified_cards[target].astype('string')

simplified_cards['cmc'] = simplified_cards['cmc'].astype('int')

Int columns — replace special string values with sentinel integers

In [182]:
int_conversion_targets = ['power', 'toughness']
unique_pt = {"nan": -2, "*": -3, "1+*": -4, "2+*": -5, "7-*": -6}

for target in int_conversion_targets:
    simplified_cards.loc[simplified_cards[target].isnull(), target] = '-2'
    simplified_cards.loc[simplified_cards[target] == '*',   target] = '-3'
    simplified_cards.loc[simplified_cards[target] == '1+*', target] = '-4'
    simplified_cards.loc[simplified_cards[target] == '*+1', target] = '-4'
    simplified_cards.loc[simplified_cards[target] == '2+*', target] = '-5'
    simplified_cards.loc[simplified_cards[target] == '*+2', target] = '-5'
    simplified_cards.loc[simplified_cards[target] == '7-*', target] = '-6'
    simplified_cards[target] = simplified_cards[target].astype('int')

Now we get the datatypes for the color identity column

In [183]:
simplified_cards.info()
print(simplified_cards['toughness'].unique())

simplified_cards.loc[simplified_cards['toughness'] == -6]

<class 'pandas.core.frame.DataFrame'>
Index: 30722 entries, 00037840-6089-42ec-8c5c-281f9f474504 to ffff90c3-63c4-4dee-a21d-6b2b113f4f80
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   object          30722 non-null  object 
 1   name            30722 non-null  string 
 2   mana_cost       30240 non-null  object 
 3   cmc             30722 non-null  int64  
 4   type_line       30722 non-null  string 
 5   oracle_text     29963 non-null  string 
 6   power           30722 non-null  int64  
 7   toughness       30722 non-null  int64  
 8   colors          30240 non-null  object 
 9   color_identity  30722 non-null  object 
 10  keywords        30722 non-null  object 
 11  all_parts       5117 non-null   object 
 12  legalities      30722 non-null  object 
 13  game_changer    30722 non-null  bool   
 14  set             30722 non-null  string 
 15  set_type        30722 non-null  string 
 16  rarity         

,object,name,mana_cost,cmc,type_line,oracle_text,power,toughness,colors,color_identity,...,set_type,rarity,watermark,card_back_id,edhrec_rank,prices,card_faces,produced_mana,loyalty,printed_text
oracle_id,,,,,,,,,,,,,,,,,,,,,
82a6d89d-9215-4540-b7d5-26cdd6afb05b,card,Shapeshifter,{6},6,Artifact Creature — Shapeshifter,"As this creature enters, choose a number betwe...",-3,-6,[],[],...,masters,uncommon,<NA>,0aeebaf5-8c7d-4636-9e82-8c27447861f7,27887.0,"{'usd': None, 'usd_foil': None, 'usd_etched': ...",NaN,NaN,NaN,<NA>


Now we will examine the colors of each card and establish a new column that is
indicative of how many colors a column has.

then we can sort for the actual colors based on a lambda expression that helps to "search" 
the list of colors that the color identity column actually looks like.

In [184]:
# To see how many colors each card has
simplified_cards['color_count'] = simplified_cards['color_identity'].str.len()

# To check if a card is "Red" or "Green" inclusive
is_white = simplified_cards['color_identity'].apply(lambda x: 'W' in x)
is_blue = simplified_cards['color_identity'].apply(lambda x: 'U' in x)
is_black = simplified_cards['color_identity'].apply(lambda x: 'B' in x)
is_red = simplified_cards['color_identity'].apply(lambda x: 'R' in x)
is_green = simplified_cards['color_identity'].apply(lambda x: 'G' in x)

cards_by_color = {'White': simplified_cards[is_white], 
                  'Blue': simplified_cards[is_blue], 
                  'Black': simplified_cards[is_black], 
                  'Red': simplified_cards[is_red], 
                  'Green': simplified_cards[is_green]}
